# 02 · Real-data profiling (Project Data Sphere)

Per-trial profiles of the five real NSCLC comparator-arm datasets used in the
revised validation design: `_438` (the training trial, the only one with
lesion-level SLD trajectories) plus the four external-validation trials
(`141`, `272`, `133`, `108`).

For each trial this shows sample size, covariate distributions, missingness,
follow-up duration, endpoint maturity, and aggregate best-overall-response
(BOR) — the Step-1 sanity checks that must pass before modelling.

> Runs on **DUA-protected** patient-level data placed under `data/raw/<trial>/`;
> nothing here is committed. See `data/DATA_SOURCES.md`. Backed by
> `vca.data_processing.pds_trials` + `vca.validation.profiling`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

from vca.data_processing.pds_trials import load_all, BOR_LEVELS
from vca.validation.profiling import profile_trial, profiles_to_frame

trials = load_all()                       # {trial_id: RealTrial}
profiles = {tid: profile_trial(rt) for tid, rt in trials.items()}
print('loaded', list(trials))

## Headline comparison (all five trials)

In [ ]:
profiles_to_frame(profiles)

## Per-trial detail

Covariate distributions, endpoint medians (Kaplan–Meier), and real BOR for each
trial. Note `_438` is the only trial with a longitudinal SLD table; `133` PFS is
not derivable from its SDTM export (OS only).

In [ ]:
def show_trial(tid):
    rt, p = trials[tid], profiles[tid]
    print(f"===== {tid}  n={p['n_patients']}  |  {p['regimen']}  |  {p['line']}  |  {p['histology_label']}  |  reader={p['reader']} =====")
    print('PFS:', p['pfs'], '\nOS :', p['os'])
    print('PFS convention:', p['pfs_convention'])
    print('OS  convention:', p['os_convention'])
    num = pd.DataFrame(p['numeric_covariates']).T[['mean', 'sd', 'pct_missing']]
    display(num)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3))
    for a, cov in zip(ax, ['histology', 'smoking', 'sex']):
        d = p['categorical_covariates'].get(cov, {})
        a.bar(list(map(str, d)), list(d.values())); a.set_title(cov); a.tick_params(axis='x', rotation=30)
    fig.tight_layout(); plt.show()
    bor = rt.bor.value_counts().reindex(list(BOR_LEVELS) + ['NE'], fill_value=0)
    (bor / bor.sum()).plot.bar(figsize=(5, 3), title=f'{tid} — best overall response (incl. NE)'); plt.ylabel('proportion'); plt.show()

for tid in trials:
    show_trial(tid)

## `_438` tumour-size (SLD) trajectories

The training trial is the only one with patient-level RECIST SLD over time.

In [ ]:
lg = trials['438'].data.longitudinal
fig, ax = plt.subplots(figsize=(7, 4))
for pid, g in lg.groupby('patient_id'):
    g = g.sort_values('time_days')
    ax.plot(g['time_days'], g['sld_mm'], color='steelblue', alpha=0.15, lw=0.8)
ax.set_xlabel('days from randomisation'); ax.set_ylabel('target-lesion SLD (mm)')
ax.set_title(f"_438 SLD trajectories (n={lg['patient_id'].nunique()} patients, {len(lg)} measurements)")
ax.set_xlim(-30, 720); plt.show()

## Benchmark sanity check

Do the real KM medians sit inside the ClinicalTrials.gov historical range for
the indication? (Simulated medians are checked separately in
`scripts/run_real_data_validation.py`.)

In [ ]:
bench = pd.read_csv('../data/benchmarks/nsclc_trial_benchmarks.csv')
band = {ep: bench.loc[bench.endpoint == ep, 'median_months'].pipe(lambda s: s[(s > 0) & (s < 60)]) for ep in ['PFS', 'OS']}
for ep in ['PFS', 'OS']:
    x = band[ep]
    print(f"{ep} benchmark: median {x.median():.1f}  p5-p95 [{x.quantile(.05):.1f}, {x.quantile(.95):.1f}] mo")
pd.DataFrame([{ 'trial': tid, 'PFS_mo': profiles[tid]['pfs']['median_months'], 'OS_mo': profiles[tid]['os']['median_months'] } for tid in trials])